# 1. ROI training data

Please download PathGen 1.6M from: https://huggingface.co/datasets/jamessyx/PathGen

Follow the instructions to process the datasets from WSI to ROI with paired summary and QA.

In [ ]:
import json
import os.path
import string
from argparse import ArgumentParser
from os.path import dirname, join, basename
from multiprocessing import Pool

from PIL import Image
from PIL.Image import Resampling
from datasets import Dataset
from tqdm import tqdm

from PIL import PngImagePlugin
# Set the limit to 100MB (100 * 1024 * 1024 bytes)
PngImagePlugin.MAX_TEXT_CHUNK = 100 * 1024 * 1024


def process_single_chunk(chunk_data):
    chunk_i, dataset_chunk, image_size, debug_image_dir, output_path = chunk_data

    with open(output_path.replace(".parquet", f".{chunk_i}.json"), "w") as f:
        json.dump(dataset_chunk, f, indent=4, ensure_ascii=False)

    processed_chunk = []
    chunk_image_sizes = []
    
    chunk_desc = f"Chunk {chunk_i}"
    pbar = tqdm(dataset_chunk, desc=chunk_desc, position=chunk_i, leave=True)
    
    for doc_index, doc in enumerate(pbar):
        try:
            image_path = doc["images"][0]
            image = Image.open(image_path).convert("RGB")
            chunk_image_sizes.append(image.size)
            
            if image_size is not None:
                image = image.resize((image_size, image_size), resample=Resampling.BICUBIC)

            clean_image = Image.new('RGB', image.size)
            clean_image.paste(image)

            processed_doc = doc.copy()
            processed_doc["images"] = [clean_image]
            processed_chunk.append(processed_doc)
            if doc_index % 1000 == 0:
                clean_image.save(join(debug_image_dir, f"chunk_{chunk_i}_{basename(image_path)}"), optimize=True)
                
            pbar.set_postfix({
                'processed': len(processed_chunk),
                'current': basename(image_path)[-10:] + '...' if len(basename(image_path)) > 10 else basename(image_path)
            })
                
        except Exception as e:
            pbar.write(f"Chunk {chunk_i} - image {doc.get('images', ['unknown'])[0]} error: {e}")
            continue
    
    pbar.close()
    
    if processed_chunk:
        chunk_output_path = output_path.replace(".parquet", f".{chunk_i}.parquet")
        chunk_dataset = Dataset.from_list(processed_chunk)
        chunk_dataset.to_parquet(chunk_output_path)
    
    return chunk_i, len(processed_chunk), chunk_image_sizes


QUESTION_TEMPLATE = """{question} 
Your task: 
1. Think through the question step by step, enclose your reasoning process in <think>...</think> tags. 
2. Then provide the correct single-letter choice (A, B, C, D,...) inside <answer>...</answer> tags.
3. No extra information or text outside of these tags."""


def process_datasets(
        json_path: str,
        image_size: int = None,
        chunk_num: int = 128,
        proc_num: int = 16,
):
    data_root = dirname(json_path)

    path_reason_ds = json.load(open(json_path))
    path_reason_ds = [doc for doc in path_reason_ds if doc["type"] == "CLOSE"]

    if image_size is None:
        output_path = join(data_root, "verl_processed_data", "original", f"{chunk_num}chunks", "path_reason_mcqa_train.parquet")
    else:
        output_path = join(data_root, "verl_processed_data", f"image{image_size}", f"{chunk_num}chunks", "path_reason_mcqa_train.parquet")

    os.makedirs(dirname(output_path), exist_ok=True)
    debug_image_dir = output_path.replace(".parquet", f"_debug")
    os.makedirs(debug_image_dir, exist_ok=True)

    all_datasets = []
    for doc_index, doc in enumerate(tqdm(path_reason_ds, desc="Processing datasets")):
        doc_image = doc["image"].replace("/", "_")
        image_path = os.path.abspath(join(data_root, "../Yale_Path/output_reason/output_reason", doc_image))
        if not os.path.exists(image_path):
            candidate_image_pattern = ".".join(image_path.split(".")[:-1])
            candidate_exist = False
            for suffix in ["png", "jpg", "jpeg"]:
                candidate_image_path = candidate_image_pattern + f".{suffix}"
                if os.path.exists(candidate_image_path):
                    candidate_exist = True
                    break
            if not candidate_exist:
                print(f"Candidate image {candidate_image_pattern} not found.")
                continue
            image_path = candidate_image_path

        conversations = doc["conversations"]
        assert len(conversations) % 2 == 0, len(conversations)

        for conv_start_idx in range(0, len(conversations), 2):
            conv_end_idx = conv_start_idx + 2

            conv_qa = conversations[conv_start_idx:conv_end_idx]
            assert conv_qa[0]["from"] == "human" and conv_qa[1]["from"] == "gpt", conv_qa
            question = conv_qa[0]["value"]
            answer = conv_qa[1]["value"]

            assert question.count("<image>") in [0, 1], conv_qa
            assert answer.strip() in string.ascii_uppercase, conv_qa
            question = "<image>\n" + question.replace("<image>", "").strip()
            question = "\n\n".join(question.split("\n\n")[:-1])

            last_option = question.split("\n")[-1].lstrip("(").split()[0]
            last_option = last_option.rstrip(")").rstrip(".").strip()
            assert last_option in string.ascii_uppercase, conv_qa

            all_datasets.append(
                {
                    "data_source": "path_reason",
                    "prompt": [
                        {"role": "user", "content": QUESTION_TEMPLATE.format(question=question)},
                    ],
                    # raw_prompt: same as "prompt" if needed
                    "images": [image_path],
                    "ability": "pathology",
                    "reward_model": {
                        "style": "rule",
                        "ground_truth": answer,
                    },
                }
            )

    print(f"train {len(all_datasets)} samples")

    # cross chunking
    all_datasets_chunks = [all_datasets[i::chunk_num] for i in range(chunk_num)]

    chunk_data_list = []
    for chunk_i in range(chunk_num):
        chunk_data_list.append((
            chunk_i,
            all_datasets_chunks[chunk_i],
            image_size,
            debug_image_dir,
            output_path
        ))

    all_image_sizes = []
    results = []
    total_processed = 0

    # multi-process version
    with Pool(processes=proc_num) as pool:
        async_result = pool.map_async(process_single_chunk, chunk_data_list)
        pool_results = async_result.get()
        
        for result in pool_results:
            chunk_i, processed_count, chunk_image_sizes = result
            all_image_sizes.extend(chunk_image_sizes)
            total_processed += processed_count
            results.append((chunk_i, processed_count))

    for chunk_i, processed_count in sorted(results):
        print(f"Chunk {chunk_i}: {processed_count} samples")

    unique_image_sizes = list(set(all_image_sizes))
    print(f"image_size_list: {unique_image_sizes}")


if __name__ == "__main__":
    parser = ArgumentParser()
    parser.add_argument("--data_root", type=int, required=True)
    parser.add_argument("--image_size", type=int, default=None)
    parser.add_argument("--chunk_num", type=int, default=128)
    args = parser.parse_args()
    process_datasets(
        json_path=f"{args.data_root}/PathReason/PathGen-Instruct.json",
        image_size=args.image_size,
        chunk_num=args.chunk_num,
    )


# 2. ROI testing data

Please downlaod information from PathMMU: https://huggingface.co/datasets/jamessyx/PathMMU

# 3. Spatial Transcriptomic data

These datasets can be accessible from these two repos:

1. HEST-1K: https://github.com/mahmoodlab/hest
2. https://github.com/JiawenChenn/STimage-1K4M 

Follow their steps and we can download the processed transcriptomic data. We then utilize the following codes to generate training/testing datasets.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
adata_train = sc.read("/home/tl688/pitl688/PathGen-1.6M/stimagedata/adata_train_gli.h5ad")
adata_valid = sc.read("/home/tl688/pitl688/PathGen-1.6M/stimagedata/adata_val_gli.h5ad")
adata_test = sc.read("/home/tl688/pitl688/PathGen-1.6M/stimagedata/adata_test_gli.h5ad")
adata_train.obs_names =[str(i) for i in range(len(adata_train))]
adata_valid.obs_names =[str(i) for i in range(len(adata_valid))]
adata_test.obs_names =[str(i) for i in range(len(adata_test))]

In [ ]:
sc.pp.normalize_total(adata_train)
sc.pp.log1p(adata_train)
sc.pp.normalize_total(adata_valid)
sc.pp.log1p(adata_valid)
sc.pp.normalize_total(adata_test)
sc.pp.log1p(adata_test)

In [ ]:
adata_train.var['means'] = np.mean(adata_train.X,axis=0)

df_var = adata_train.var

df_var.sort_values('means', ascending=False)

In [ ]:
def generate_instruct(adata, tissue='gliomas'):
    adata.var['means'] = np.mean(adata.X,axis=0)
    df_var = adata.var
    ranked_geen_info = df_var.sort_values('means', ascending=False)
    instruct = []
    output = []
    for rangex in range(len(adata)):
        instruct.append(f"Generate top 100 genes from one spot shown in the histopathology image in {tissue} ranked by expression level.")
        ranked_gene = list(adata.var_names[np.argsort(adata[rangex,:].X[0])[::-1]][0:100])
        output.append(ranked_gene)
    
    return pd.DataFrame({'instruct':instruct, 'output':output})
out1 = generate_instruct(adata_train)
out1.to_csv("./image_info/train_gliomas_out.csv")
out1 = generate_instruct(adata_valid)
out1.to_csv("./image_info/valid_gliomas_out.csv")
out1 = generate_instruct(adata_test)
out1.to_csv("./image_info/test_gliomas_out.csv")

In [ ]:
gene_train_token = np.unique(list(adata_train.var_names) + list(adata_valid.var_names))

df_token = pd.DataFrame(gene_train_token)
df_token.columns = ['genes']

df_token.to_csv("./image_info/gliomas_gene_token.csv")

In [ ]:
import scanpy as sc
import pandas as pd

from PIL import Image

from PIL import PngImagePlugin
LARGE_ENOUGH_NUMBER = 100
PngImagePlugin.MAX_TEXT_CHUNK = LARGE_ENOUGH_NUMBER * (1024**2)

df1 = pd.read_csv("./image_info/train_gliomas_out.csv")
df2 = pd.read_csv("./image_info/valid_gliomas_out.csv")

df_joint = pd.concat((df1,df2))

df_joint.values[0]

len(df_joint)

num_genes = 100
isntruct_set = [
        f"Generate a list of {num_genes} genes in order of descending expression from one spot shown in the histopathology image in IDC disease.\nCell sentence:",
        f"Produce a list of {num_genes} gene names in descending order of expression which represent the expressed genes from one spot shown in the histopathology image in IDC disease.\nCell sentence:",
        f"Create a ranked list of {num_genes} genes in decreasing order of expression from one spot shown in the histopathology image in IDC disease.\nCell sentence:",
        f"List the top {num_genes} expressed genes from one spot shown in the histopathology image in IDC disease.\nCell sentence:",
        f"Identify the highest expressed {num_genes} genes in decreasing order of expression from one spot shown in the histopathology image in IDC disease.\nCell sentence:",
        f"Enumerate a list of {num_genes} genes in descending order of expression from one spot shown in the histopathology image in IDC disease.\nCell sentence:",
        f"Compile a descending order list of {num_genes} expressed genes from one spot shown in the histopathology image in IDC disease.\nCell sentence:",
        f"Present a sequence of {num_genes} genes ordered by decreasing expression level from one spot shown in the histopathology image in IDC disease.\nCell sentence:",
        f"Generate an ordered list of {num_genes} genes by decreasing expression level from one spot shown in the histopathology image in IDC disease.\nCell sentence:",
        f"Assemble a list of {num_genes} genes from highest to lowest expression from one spot shown in the histopathology image in IDC disease.\nCell sentence:"
]

for i in isntruct_set:
    print(i)



df_joint

df_find_example = pd.read_json("../LLaMA-Factory/data/mllm_demo.json")

df_find_example['messages'].values[0]

df_find_example['images']



df_joint_new = pd.DataFrame()

import numpy as np

message_list = []
image_list = []
np.random.seed(2024)
for item in range(len(df_joint)):
    row_item = df_joint.iloc[item]
    guide_prompt = np.random.choice(isntruct_set,size=1)[0]
    ques = {'content':'<image>'+guide_prompt, 'role':'user'}
    answer_list = ' '.join(eval(row_item['output']))
    
    ans = {'content':answer_list, 'role':'assistant'}
    message_list.append([ques,ans])
    
    image_list.append([f'gliomas_train_image/{item}.png'])

df_joint_new['messages'] = message_list 
df_joint_new['images'] = image_list

df_joint_new.to_json("./gliomas_train.json",orient='records', indent=2)